In [5]:
import requests
r = requests.get("https://swapi-api.hbtn.io/api/starships/")
print(r.headers["Content-Type"])


application/json


In [6]:
response = requests.get("https://swapi-api.hbtn.io/api/starships/")
type(response)

requests.models.Response

In [7]:
response.status_code
response.headers
response.text
response.content
response.json()
response.url
response.ok

True

In [8]:
#!/usr/bin/env python3
"""
Method that returns the list of ships that can hold
a given number of passengers, using the SWAPI API.
"""
import requests


def availableShips(passengerCount):
    """
    Returns the list of ship names that can hold at least
    passengerCount passengers.

    If no ship is available, returns an empty list.
    """
    ships = []
    url = "https://swapi-api.hbtn.io/api/starships/"

    while url:
        response = requests.get(url)
        data = response.json()

        for ship in data.get("results", []):
            passengers = ship.get("passengers", "0").replace(",", "")
            try:
                if int(passengers) >= passengerCount:
                    ships.append(ship.get("name"))
            except ValueError:
                # passengers value is "n/a" or "unknown" - skip it
                continue

        url = data.get("next")

    return ships

In [9]:
ships = availableShips(4)
for ship in ships:
    print(ship)


CR90 corvette
Sentinel-class landing craft
Death Star
Millennium Falcon
Executor
Rebel transport
Slave 1
Imperial shuttle
EF76 Nebulon-B escort frigate
Calamari Cruiser
Republic Cruiser
Droid control ship
Scimitar
J-type diplomatic barge
AA-9 Coruscant freighter
Republic Assault ship
Solar Sailer
Trade Federation cruiser
Theta-class T-2c shuttle
Republic attack cruiser


In [10]:
#!/usr/bin/env python3
"""
Method that returns the list of names of the home planets
of all sentient species, using the SWAPI API.
"""
import requests


def sentientPlanets():
    """
    Returns the list of home planet names for every species
    whose classification or designation is 'sentient'.
    """
    planets = []
    url = "https://swapi-api.hbtn.io/api/species/"

    while url:
        response = requests.get(url)
        data = response.json()

        for species in data.get("results", []):
            classification = species.get("classification", "")
            designation = species.get("designation", "")

            if classification == "sentient" or designation == "sentient":
                homeworld_url = species.get("homeworld")
                if homeworld_url is None:
                    continue

                planet_response = requests.get(homeworld_url)
                planet_data = planet_response.json()
                planets.append(planet_data.get("name"))

        url = data.get("next")

    return planets

In [13]:

planets = sentientPlanets()
for planet in planets:
    print(planet)

Coruscant
Kashyyyk
Rodia
Nal Hutta
unknown
Trandosha
Mon Cala
Endor
Sullust
Cato Neimoidia
Naboo
Toydaria
Malastare
Ryloth
Aleen Minor
Vulpter
Troiken
Tund
Cerea
Glee Anselm
Iridonia
Tholoth
Iktotch
Quermia
Dorin
Champala
Geonosis
Mirial
Zolan
Ojom
Kamino
Skako
Muunilinst
Shili
Kalee
Utapau


In [24]:

%%writefile 2-user_location.py
#!/usr/bin/env python3
"""
Script that prints the location of a specific GitHub user,
given the full API URL as the first command line argument.
"""
import sys
import time
import requests


if __name__ == '__main__':

    if len(sys.argv) < 2:
        print("Usage: ./2-user_location.py <GitHub API user URL>")
        sys.exit(1)

    url = sys.argv[1]
    response = requests.get(url)

    if response.status_code == 404:
        print("Not found")

    elif response.status_code == 403:
        reset_timestamp = int(response.headers.get("X-Ratelimit-Reset", 0))
        current_timestamp = time.time()
        minutes = int((reset_timestamp - current_timestamp) / 60)
        print("Reset in {} min".format(minutes))

    else:
        data = response.json()
        print(data.get("location"))

Overwriting 2-user_location.py


In [25]:
!chmod +x 2-user_location.py

In [26]:
!python3 2-user_location.py https://api.github.com/users/holbertonschool

None


In [27]:
!python3 2-user_location.py https://api.github.com/users/octocat

San Francisco


In [ ]:
#!/usr/bin/env python3
"""
Script that displays the first (soonest) upcoming SpaceX launch:
name, local date, rocket name, and launchpad name + locality.
"""
import requests


if __name__ == '__main__':
    # /upcoming gives us every launch that hasn't happened yet.
    url = "https://api.spacexdata.com/v4/launches/upcoming"
    launches = requests.get(url).json()

    # Sort by date_unix (the raw timestamp, safe to compare as a
    # number) so the soonest launch ends up first. Python's sort
    # is stable, so if two launches share the exact same date_unix,
    # whichever came first in the API response stays first here -
    # exactly what the task asks for.
    launches.sort(key=lambda launch: launch["date_unix"])
    first_launch = launches[0]

    launch_name = first_launch["name"]
    date_local = first_launch["date_local"]

    # rocket/launchpad on the launch object are just ID strings,
    # not the actual data - need a follow-up request for each,
    # same pattern as fetching a species' homeworld in task 1.
    rocket_id = first_launch["rocket"]
    rocket_url = "https://api.spacexdata.com/v4/rockets/{}".format(
        rocket_id)
    rocket_name = requests.get(rocket_url).json()["name"]

    launchpad_id = first_launch["launchpad"]
    launchpad_url = "https://api.spacexdata.com/v4/launchpads/{}".format(
        launchpad_id)
    launchpad_data = requests.get(launchpad_url).json()
    launchpad_name = launchpad_data["name"]
    launchpad_locality = launchpad_data["locality"]

    print("{} ({}) {} - {} ({})".format(
        launch_name,
        date_local,
        rocket_name,
        launchpad_name,
        launchpad_locality
    ))